In [2]:
%%capture
!pip install -U sagemaker
!pip install seaborn

## Dependencies

In [31]:
import numpy as np
from sagemaker import get_execution_role
import sagemaker
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.tuner import (
    IntegerParameter,
    CategoricalParameter,
    ContinuousParameter,
    HyperparameterTuner,
)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
import datetime
import time
from time import gmtime, strftime
import tarfile
import boto3
import pandas as pd

## Setup

In [4]:
sm_boto3 = boto3.client("sagemaker")
sess = sagemaker.Session()
region = sess.boto_session.region_name
bucket = "sklearn-mlops-tutorial"
print("Using bucket " + bucket)

Using bucket sklearn-mlops-tutorial


In [5]:
df = pd.read_csv("/root/SKLearn/data/mob_price_classification_train.csv")

## Process data

In [6]:
features = list(df.columns)
label = features.pop(-1)

In [7]:
x = df[features]
y = df[label]

#### Split Data

In [8]:
X_train, X_test, y_train, y_test = train_test_split(x,y, test_size=0.15, random_state=0)

In [9]:
trainX = pd.DataFrame(X_train)
trainX[label] = y_train

testX = pd.DataFrame(X_test)
testX[label] = y_test

In [10]:
trainX.to_csv("/root/SKLearn/processed_data/train-V-1.csv",index = False)
testX.to_csv("/root/SKLearn/processed_data/test-V-1.csv", index = False)

#### Upload data to S3

In [11]:
# send data to S3. SageMaker will take training data from s3
sk_prefix = "sagemaker/mobile_price_classification/sklearncontainer"

trainpath = sess.upload_data(
    path="/root/SKLearn/processed_data/train-V-1.csv", bucket=bucket, key_prefix=sk_prefix
)

testpath = sess.upload_data(
    path="/root/SKLearn/processed_data/test-V-1.csv", bucket=bucket, key_prefix=sk_prefix
)

In [12]:
# # test the script locally
# ! python code/script.py --n_estimators 100 \
#                    --random_state 0 \
#                    --model-dir ./ \
#                    --train ./ \
#                    --test ./ \


## Deploy training job using SKLearn container

In [66]:
FRAMEWORK_VERSION = "0.23-1"

sklearn_estimator = SKLearn(
    source_dir="code",
    entry_point="script.py",
    role=get_execution_role(),
    instance_count=1,
    instance_type="ml.m5.large",
    framework_version=FRAMEWORK_VERSION,
    base_job_name="RF-custom-sklearn",
    use_spot_instances = True,
    max_wait = 7200,
    max_run = 3600
)

In [67]:
# hyperparameters={
#         "n_estimators": 100,
#         "random_state": 0,
#     },

In [68]:
hyperparameter_ranges = {
    "n_estimators": IntegerParameter(1, 100, scaling_type="Linear"),
}

objective_metric_name = "accuracy"
objective_type = "Maximize"
metric_definitions = [{"Name": "accuracy", "Regex": "Model Accuracy is: ([0-9\\.]+)"}]
# .*train:loss ([0-9\\.]+).*

In [69]:
tuner_log = HyperparameterTuner(
    sklearn_estimator,
    objective_metric_name,
    hyperparameter_ranges,
    metric_definitions,
    max_jobs=3,
    max_parallel_jobs=3,
    strategy="Random",
    objective_type=objective_type
)

In [70]:
tuner_log.fit(
    {"train": trainpath, "test": testpath},
    include_cls_metadata=False,
    job_name="sklearn-" + strftime("%Y%m%d-%H-%M-%S", gmtime()),
)

[12/27/24 18:38:22] WARNING  No finished training job found associated with this estimator.       ]8;id=560612;file:///opt/conda/lib/python3.11/site-packages/sagemaker/estimator.py\estimator.py]8;;\:]8;id=784854;file:///opt/conda/lib/python3.11/site-packages/sagemaker/estimator.py#1914\1914]8;;\
                             Please make sure this estimator is only used for building workflow                    
                             config                                                                                

                    INFO     Creating hyperparameter tuning job with name:                          ]8;id=903588;file:///opt/conda/lib/python3.11/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=877235;file:///opt/conda/lib/python3.11/site-packages/sagemaker/session.py#3383\3383]8;;\
                             sklearn-20241227-18-38-22                                                             

...................................!


## Analyse Tuner

In [77]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

# check jobs have finished
status_log = boto3.client("sagemaker").describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=tuner_log.latest_tuning_job.job_name
)["HyperParameterTuningJobStatus"]
# status_linear = boto3.client("sagemaker").describe_hyper_parameter_tuning_job(
#     HyperParameterTuningJobName=tuner_linear.latest_tuning_job.job_name
# )["HyperParameterTuningJobStatus"]

assert status_log == "Completed", "First must be completed, was {}".format(status_log)
# assert status_linear == "Completed", "Second must be completed, was {}".format(status_linear)

df_log = sagemaker.HyperparameterTuningJobAnalytics(
    tuner_log.latest_tuning_job.job_name
).dataframe()
# df_linear = sagemaker.HyperparameterTuningJobAnalytics(
#     tuner_linear.latest_tuning_job.job_name
# ).dataframe()
df_log["scaling"] = "log"
# df_linear["scaling"] = "linear"
# df = pd.concat([df_log, df_linear], ignore_index=True)


In [ ]:
# identify best model
df_log.head()